In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import col, explode_outer, to_timestamp, get_json_object, from_json
from pyspark.sql.types import ArrayType, StructType, StructField, StringType


@dp.table(
    name="gharchive_silver",
    comment="Flattened GitHub Archive events with exploded commits"
)
def gharchive_silver():
    # Streaming read from bronze — silver also becomes a streaming table
    df = spark.readStream.table("gharchive_bronze")

    # Flatten repo struct
    df = (df
        .withColumn("repo_id", col("repo.id"))
        .withColumn("repo_name", col("repo.name"))
        .withColumn("repo_url", col("repo.url"))
    )

    # Flatten actor struct
    df = (df
        .withColumn("actor_id", col("actor.id"))
        .withColumn("actor_login", col("actor.login"))
        .withColumn("actor_display_login", col("actor.display_login"))
        .withColumn("actor_avatar_url", col("actor.avatar_url"))
    )

    # Extract commit fields from payload JSON string
    # payload is stored as raw JSON string in bronze to avoid schema merge conflicts
    COMMIT_SCHEMA = ArrayType(StructType([
        StructField("sha", StringType()),
        StructField("message", StringType()),
        StructField("author", StructType([
            StructField("name", StringType()),
            StructField("email", StringType()),
        ])),
    ]))

    commits_json = get_json_object(col("payload"), "$.commits")
    df = df.withColumn("_commits", from_json(commits_json, COMMIT_SCHEMA))

    # Explode commits (only PushEvents have them, others will be null)
    df = df.withColumn("commit", explode_outer("_commits"))
    df = (df
        .withColumn("commit_sha", col("commit.sha"))
        .withColumn("commit_message", col("commit.message"))
        .withColumn("commit_author_name", col("commit.author.name"))
        .withColumn("commit_author_email", col("commit.author.email"))
    )
    df = df.drop("commit", "_commits")

    # Extract action from payload (common across event types)
    df = df.withColumn("payload_action", get_json_object(col("payload"), "$.action"))

    # Parse created_at to timestamp
    df = df.withColumn("created_at", to_timestamp("created_at"))

    return df.drop("repo", "actor", "payload", "org", "_rescued_data")